# Production Simulation on Kaggle

Dua mode, pilih lewat `SOURCE_MODE`:

1. **`live`** - rekam HLS 30 detik -> proses -> replay, diulang `LIVE_CYCLES` kali.
   Ini mode default.
2. **`drive_video`** - unduh satu video 1:02:26 dari Google Drive, proses per 30 detik,
   lalu replay tiap window ke pipeline.

**Alur:** cek lingkungan -> clone/install -> set secret -> jalankan mode -> unduh bundle.

Hasil dikirim ke pipeline produksi lewat **Cloudflare Tunnel**
(`https://lane-api.syaifudin-fam.my.id`), bukan ngrok. Dashboard:
`https://lane.syaifudin-fam.my.id`.

> Kaggle: Settings -> Accelerator -> **GPU T4 x2** (opsional), Internet **ON**.
> Default pipeline adalah **single pass**. Untuk anotasi satu video dengan 2xT4,
> gunakan `notebooks/colab_annotate_tmp_kaggle.ipynb`.


In [ ]:
import shutil

import torch

print("torch      :", torch.__version__)
print("CUDA ready :", torch.cuda.is_available())
print("GPU count  :", torch.cuda.device_count())
for index in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(index)
    print(f"  GPU {index}    : {props.name} ({props.total_memory / 1e9:.1f} GB)")

print("ffmpeg     :", shutil.which("ffmpeg"))
print("ffprobe    :", shutil.which("ffprobe"))

from pathlib import Path

for folder in ("/kaggle/input", "/kaggle/working"):
    if Path(folder).exists():
        usage = shutil.disk_usage(folder)
        print(f"{folder}: {usage.free / 1e9:.1f} GB free")

## 1. Clone dan install

Kaggle sudah membawa PyTorch CUDA dan ffmpeg; jangan install wheel torch CPU.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO = "https://github.com/ibrahim119920/highway-lane-analytics.git"
PROJECT = "highway-lane-occupancy"

if not Path(PROJECT).is_dir():
    subprocess.run(["git", "clone", REPO, PROJECT], check=True)
os.chdir(PROJECT)
# Always pick up the pushed tuning/pipeline changes; a dirty working tree only
# produces a warning instead of stopping the notebook.
subprocess.run(["git", "-C", PROJECT, "pull", "--ff-only"], check=False)

ROOT = Path.cwd()
SRC = ROOT / "src"
assert (SRC / "lane_analytics").is_dir(), f"Repo tidak lengkap: {SRC}"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[colab]"],
    check=True,
)

import ultralytics
from lane_analytics.colab import activate
from lane_analytics.config import as_plain_dict
from lane_analytics.pipeline import process_video
from lane_analytics.replay import replay_run

print("repo         :", ROOT)
print("ultralytics  :", ultralytics.__version__)
print("activate     :", activate.__module__)

## 2. Pilih mode dan parameter

`SOURCE_MODE = "live"` menjalankan opsi 1 (HLS). `SOURCE_MODE = "drive_video"`
menjalankan opsi 2 (video Drive 1:02:26 per 30 detik).

Untuk mengirim ke pipeline produksi lewat Cloudflare:

1. Tambahkan secret **`PIPELINE_WEBHOOK_TOKEN`** di **Add-ons -> Secrets**.
   Opsional: `PIPELINE_BASE_URL` bila hostname berubah.
2. Set `REPLAY = True` (default).
3. `DRY_RUN = True` untuk menulis batch ke `output/replay/` tanpa HTTP.
   `DRY_RUN = False` untuk benar-benar POST ke pipeline.
4. `REPLAY = False` untuk melewati replay sepenuhnya.

Token tidak pernah ditulis di notebook. Bila secret tidak ada, replay HTTP dilewati
dengan peringatan; dry-run tetap bisa dipakai untuk memeriksa payload.


In [ ]:
#@title Mode dan parameter { display-mode: "form" }
SOURCE_MODE = "live"  #@param ["live", "drive_video"]

# --- opsi 1: live HLS ------------------------------------------------------
LIVE_URL = "https://pub2.hk-opt.com/LiveApp/streams/501264524429471982550467.m3u8"
LIVE_CYCLES = 3            # jumlah window rekam -> proses -> replay
RECORD_SECONDS = 30        # durasi rekaman per window
CYCLE_SLEEP_SECONDS = 0.0  # jeda antar window (0 = langsung lanjut)

# --- opsi 2: satu video Drive 1:02:26 -------------------------------------
DRIVE_FILE_ID = "1mgunh3mQX8YYM4doPSNSSG1fITkQvvFd"
DRIVE_URL = f"https://drive.google.com/uc?id={DRIVE_FILE_ID}"
DRIVE_VIDEO_PATH = "/kaggle/working/source_62min.mp4"
WINDOW_SECONDS = 30        # olah per 30 detik
MAX_WINDOWS = 0            # 0 = semua (~124 window)
# GANTI dengan waktu overlay frame 0 video. Wajib untuk mode drive_video.
BASE_TIMESTAMP = "2026-11-08T00:00:00"

# --- model, waktu, dan replay ---------------------------------------------
SESSION_TIMEZONE = "Asia/Jakarta"  # live memakai WIB, bukan UTC container
FRAME_STRIDE = 1
IMGSZ = 960
REPLAY = True              # True = kirim ke pipeline
DRY_RUN = False            # True = tulis batch, tanpa HTTP
SPEED = 5.0                # 1 = real time, 5 = 5x
REPLAY_WINDOW_SECONDS = 5.0
SKIP_HEALTH_CHECK = False  # True hanya untuk receiver tanpa /health
HEALTH_ATTEMPTS = 6        # poll /health sampai Kafka siap
HEALTH_DELAY = 10.0        # detik antar poll
# ---------------------------------------------------------------------------

assert SOURCE_MODE in {"live", "drive_video"}, SOURCE_MODE
print("mode          :", SOURCE_MODE)
print("timezone      :", SESSION_TIMEZONE)
print("replay        :", REPLAY, "dry_run:", DRY_RUN)
if SOURCE_MODE == "live":
    print("live url      :", LIVE_URL)
    print("windows       :", LIVE_CYCLES, "x", RECORD_SECONDS, "s")
else:
    print("drive video   :", DRIVE_URL)
    print("window        :", WINDOW_SECONDS, "s, max:", MAX_WINDOWS or "all")
    print("base timestamp:", BASE_TIMESTAMP)
print("skip health   :", SKIP_HEALTH_CHECK)


## 3. Secret dan config efektif

Config asli disalin ke `/kaggle/working/config_kaggle.yaml` dengan evidence dimatikan,
sehingga setiap siklus/window tidak membuat snapshot dan klip 8 detik.

In [ ]:
from pathlib import Path

import yaml

from lane_analytics.utils import setup_logging

setup_logging(verbose=False)

# Pipeline produksi lewat Cloudflare Tunnel. Jangan menulis token di notebook.
PIPELINE_URL = "https://lane-api.syaifudin-fam.my.id"
# Production write token, published on purpose so a fresh Kaggle session can send.
# Rotate it on the server if it is abused.
PIPELINE_TOKEN = "86bdedb9d0f8610dc53df356d98d853adb720df5baccd59d"
WEBHOOK_URL = PIPELINE_URL
WEBHOOK_TOKEN = PIPELINE_TOKEN
try:
    from kaggle_secrets import UserSecretsClient

    _secrets = UserSecretsClient()
    WEBHOOK_URL = _secrets.get_secret("PIPELINE_BASE_URL") or PIPELINE_URL
    WEBHOOK_TOKEN = _secrets.get_secret("PIPELINE_WEBHOOK_TOKEN") or PIPELINE_TOKEN
except Exception as exc:
    print("Kaggle Secrets tidak tersedia, memakai token bawaan:", type(exc).__name__)

DO_REPLAY = bool(REPLAY and (DRY_RUN or WEBHOOK_TOKEN))
if REPLAY and not DO_REPLAY:
    print(
        "warning: PIPELINE_WEBHOOK_TOKEN kosong; replay HTTP dilewati. "
        "Set DRY_RUN=True untuk uji payload, atau tambahkan secret."
    )

CONFIG_PATH = Path("/kaggle/working/config_kaggle.yaml")
raw = yaml.safe_load(Path("config/config.yaml").read_text(encoding="utf-8"))
raw["evidence"]["enabled"] = False
raw["video"]["frame_stride"] = FRAME_STRIDE
raw["model"]["imgsz"] = IMGSZ
CONFIG_PATH.write_text(yaml.safe_dump(raw, sort_keys=False), encoding="utf-8")

print("pipeline      :", WEBHOOK_URL)
print("token         :", "set" if WEBHOOK_TOKEN else "empty")
print("do replay     :", DO_REPLAY)
print("config        :", CONFIG_PATH)
print("logging       : INFO (progress rekaman/proses/replay akan tampil)")


## 4. Opsi 1 - live HLS

Setiap siklus:

1. `activate(source="live")` merekam `RECORD_SECONDS` detik dari `LIVE_URL`.
2. Pipeline single-pass mendeteksi kendaraan, lane events, dan hogging.
3. Events/tracks di-replay ke webhook (atau dry-run).
4. Siklus diulang `LIVE_CYCLES` kali.

Jalankan cell ini hanya jika `SOURCE_MODE = "live"`.

In [ ]:
import time
import traceback
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

live_results = []

if SOURCE_MODE == "live":
    device = "0" if torch.cuda.is_available() else "cpu"
    session_started = datetime.now(ZoneInfo(SESSION_TIMEZONE)).replace(tzinfo=None)
    session_id = f"live_{session_started:%Y%m%d_%H%M%S}"

    print(f"session_id     : {session_id}")
    print(f"session start  : {session_started.isoformat()} ({SESSION_TIMEZONE})")
    print(f"windows        : {LIVE_CYCLES} x {RECORD_SECONDS}s")

    for cycle in range(1, int(LIVE_CYCLES) + 1):
        window_index = cycle - 1
        run_id = f"{session_id}_w{window_index:04d}"
        base_ts = (
            session_started + timedelta(seconds=window_index * RECORD_SECONDS)
        ).isoformat()

        print(f"\n=== LIVE WINDOW {cycle}/{LIVE_CYCLES} | {run_id} ===")
        print(f"stage: record {RECORD_SECONDS}s -> process -> replay")
        print(f"base_timestamp: {base_ts}")

        started = time.time()
        try:
            result = activate(
                source="live",
                stream_url=LIVE_URL,
                record_seconds=RECORD_SECONDS,
                video_dir="data/raw/live",
                config_path=CONFIG_PATH,
                device=device,
                imgsz=IMGSZ,
                frame_stride=FRAME_STRIDE,
                write_video=False,
                replay=DO_REPLAY,
                webhook_url=WEBHOOK_URL or None,
                webhook_token=WEBHOOK_TOKEN or None,
                replay_source="kaggle-replay",
                speed=SPEED,
                window_seconds=REPLAY_WINDOW_SECONDS,
                dry_run=DRY_RUN,
                skip_health_check=SKIP_HEALTH_CHECK,
                health_attempts=HEALTH_ATTEMPTS,
                health_delay=HEALTH_DELAY,
                session_id=session_id,
                window_index=window_index,
                window_count=LIVE_CYCLES,
                run_id=run_id,
                base_timestamp=base_ts,
            )
            live_results.append(result)
            print(result)
            for item in result.videos:
                print(
                    f"  {item['video']} | status={item['status']} | "
                    f"events={item.get('events', '-')} | vehicles={item.get('vehicles', '-')} | "
                    f"session={item.get('session_id')} "
                    f"window={item.get('window_index')}/{item.get('window_count')}"
                )
                if item.get("error"):
                    print("  error:", item["error"])
        except Exception as exc:
            print("window failed:", type(exc).__name__, exc)
            traceback.print_exc()
            live_results.append(
                {
                    "status": "error",
                    "error": str(exc),
                    "session_id": session_id,
                    "window_index": window_index,
                }
            )

        wall = time.time() - started
        print(f"window wall: {wall:.1f}s")
        if CYCLE_SLEEP_SECONDS > 0:
            time.sleep(CYCLE_SLEEP_SECONDS)

    print(f"\nsession selesai: {session_id} | windows={len(live_results)}")
else:
    print("SOURCE_MODE bukan 'live'; lewati cell ini.")


## 5. Opsi 2 - satu video Drive 1:02:26, per 30 detik

Video diunduh sekali, lalu diproses dalam window 30 detik. Setiap window:

1. `process_video(start_frame=..., frame_limit=...)` - hanya 30 detik sumber.
2. Events/tracks ditulis ke `output/`.
3. Replay window tersebut ke webhook (atau dry-run).
4. Jika proses + replay selesai sebelum 30 detik, notebook menunggu sisanya agar
   ritme mendekati real-time.

Catatan: tracker dan track ID di-reset tiap window. Pipeline penerima harus memakai
`run_id` per window dan tidak mengharapkan ID kontinu antar window. Jalankan cell ini
hanya jika `SOURCE_MODE = "drive_video"`.

In [ ]:
import math
import time
import traceback
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

import cv2
import gdown

from lane_analytics.config import load_config

drive_runs = []

if SOURCE_MODE == "drive_video":
    if not BASE_TIMESTAMP:
        raise ValueError("BASE_TIMESTAMP wajib diisi untuk mode drive_video")

    video_path = Path(DRIVE_VIDEO_PATH)
    if not video_path.exists():
        gdown.download(DRIVE_URL, str(video_path), quiet=False, fuzzy=True)
    if not video_path.exists() or video_path.stat().st_size == 0:
        raise FileNotFoundError(f"Download gagal: {video_path}")

    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        raise RuntimeError(f"Video tidak bisa dibuka: {video_path}")
    fps = float(capture.get(cv2.CAP_PROP_FPS) or 0.0)
    total_frames = int(capture.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    capture.release()
    if fps <= 0 or total_frames <= 0:
        raise RuntimeError("FPS/frame count tidak terbaca")

    window_frames = max(1, int(round(WINDOW_SECONDS * fps)))
    total_windows = int(math.ceil(total_frames / window_frames))
    if MAX_WINDOWS:
        total_windows = min(total_windows, int(MAX_WINDOWS))

    session_id = (
        f"drive_{video_path.stem}_"
        f"{datetime.now(ZoneInfo(SESSION_TIMEZONE)):%Y%m%d_%H%M%S}"
    )
    print("session_id   :", session_id)
    print("video        :", video_path, f"({video_path.stat().st_size / 1e6:.1f} MB)")
    print("fps/frames   :", round(fps, 3), "/", total_frames)
    print("window frames:", window_frames)
    print("windows      :", total_windows)

    device = "0" if torch.cuda.is_available() else "cpu"
    for index in range(total_windows):
        start = index * window_frames
        limit = min(window_frames, total_frames - start)
        run_id = f"{session_id}_w{index:04d}"
        print(
            f"\n=== WINDOW {index + 1}/{total_windows} | {run_id} | "
            f"frames {start}-{start + limit} ==="
        )

        config = load_config(CONFIG_PATH)
        config.video.path = str(video_path)
        config.video.frame_stride = FRAME_STRIDE
        config.video.max_frames = None
        config.base_timestamp = BASE_TIMESTAMP
        config.model.device = device
        config.model.imgsz = IMGSZ
        config.evidence.enabled = False

        started = time.time()
        try:
            batch = process_video(
                config,
                start_frame=start,
                frame_limit=limit,
                write_video=False,
                generate_reports=False,
                run_id=run_id,
                session_id=session_id,
                window_index=index,
                window_count=total_windows,
            )
            replay_info = None
            if DO_REPLAY:
                # summary/config dikirim dari memori, jadi /webhook/run tetap
                # terkirim walau generate_reports=False.
                replay_info = replay_run(
                    batch.events_path,
                    tracks_path=batch.tracks_path,
                    summary_data=batch.summary,
                    config_data=as_plain_dict(config),
                    url=WEBHOOK_URL or None,
                    token=WEBHOOK_TOKEN or None,
                    run_id=run_id,
                    source="kaggle-replay",
                    speed=SPEED,
                    window_seconds=REPLAY_WINDOW_SECONDS,
                    dry_run=DRY_RUN,
                    skip_health_check=SKIP_HEALTH_CHECK,
                    health_attempts=HEALTH_ATTEMPTS,
                    health_delay=HEALTH_DELAY,
                    dry_run_dir="output/replay",
                ).as_dict()

            wall = time.time() - started
            item = {
                "session_id": session_id,
                "window": index,
                "start_frame": start,
                "frames": limit,
                "run_id": run_id,
                "events": batch.total_events,
                "vehicles": batch.total_vehicles,
                "hogging": batch.hogging_events,
                "wall_seconds": round(wall, 2),
                "replay": replay_info,
            }
            drive_runs.append(item)
            print(
                f"events={item['events']} vehicles={item['vehicles']} "
                f"hogging={item['hogging']} wall={wall:.1f}s"
            )
        except Exception as exc:
            wall = time.time() - started
            print("window failed:", type(exc).__name__, exc)
            traceback.print_exc()
            drive_runs.append(
                {
                    "session_id": session_id,
                    "window": index,
                    "start_frame": start,
                    "run_id": run_id,
                    "status": "error",
                    "error": str(exc),
                    "wall_seconds": round(wall, 2),
                }
            )

        # Jaga ritme 30 detik bila proses + replay lebih cepat.
        if wall < WINDOW_SECONDS:
            time.sleep(WINDOW_SECONDS - wall)
        else:
            print(f"behind schedule by {wall - WINDOW_SECONDS:.1f}s")

    print(f"\nsession selesai: {session_id} | windows={len(drive_runs)}")
else:
    print("SOURCE_MODE bukan 'drive_video'; lewati cell ini.")


## 6. Package hasil

Bundle berisi `manifest.json` + seluruh `output/` (events, tracks, replay dry-run).
Tidak ada video anotasi di mode production ini agar hemat disk dan waktu.
Unduh dari tab **Output** Kaggle sebelum sesi berakhir.

In [ ]:
import json
import zipfile
from pathlib import Path

if SOURCE_MODE == "live":
    runs = [
        item.as_dict() if hasattr(item, "as_dict") else item
        for item in live_results
    ]
else:
    runs = drive_runs

pipeline_stats = None
if DO_REPLAY and not DRY_RUN:
    try:
        from lane_analytics.replay import fetch_stats

        pipeline_stats = fetch_stats(WEBHOOK_URL, token=WEBHOOK_TOKEN)
        print("pipeline /stats:", pipeline_stats)
    except Exception as exc:
        print("warning: /stats tidak terbaca:", type(exc).__name__, exc)

manifest = {
    "source_mode": SOURCE_MODE,
    "session_id": globals().get("session_id", ""),
    "settings": {
        "frame_stride": FRAME_STRIDE,
        "imgsz": IMGSZ,
        "replay": REPLAY,
        "dry_run": DRY_RUN,
        "speed": SPEED,
        "pipeline_url": WEBHOOK_URL if DO_REPLAY else None,
        "live_url": LIVE_URL if SOURCE_MODE == "live" else None,
        "live_cycles": LIVE_CYCLES if SOURCE_MODE == "live" else None,
        "record_seconds": RECORD_SECONDS if SOURCE_MODE == "live" else None,
        "drive_file_id": DRIVE_FILE_ID if SOURCE_MODE == "drive_video" else None,
        "window_seconds": WINDOW_SECONDS if SOURCE_MODE == "drive_video" else None,
        "base_timestamp": BASE_TIMESTAMP if SOURCE_MODE == "drive_video" else None,
    },
    "pipeline_stats": pipeline_stats,
    "runs": runs,
}

out = Path("output")
work_dir = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path("/content")
manifest_path = work_dir / "production_manifest.json"
bundle = work_dir / "production_bundle.zip"
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")

with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(manifest_path, "manifest.json")
    for path in sorted(out.rglob("*")):
        if path.is_file():
            zf.write(path, path.relative_to(out.parent))

print(f"Manifest : {manifest_path}")
print(f"Bundle   : {bundle} ({bundle.stat().st_size / 1e6:.1f} MB)")
print(f"Runs     : {len(runs)}")


## 7. Catatan production

- **Jalur data**: Kaggle -> HTTPS -> Cloudflare Tunnel -> receiver -> Kafka -> dashboard.
  Tidak perlu install `cloudflared` di Kaggle. Selalu pakai `https://`.
- **Token** hanya dari Secrets (`PIPELINE_WEBHOOK_TOKEN`). Jangan hard-code atau
  mencetak nilainya. Bila bocor, minta operator merotasinya.
- **Urutan kirim** per window: semua batch `/webhook` (urut `video_time_s`), lalu
  `/webhook/tracks` sekali, lalu `/webhook/run` dengan `complete: true` terakhir.
- **Format waktu**: `timestamp` event = WIB naive tanpa `Z`/offset. `sent_at` = UTC.
  Sender menolak timestamp ber-offset sebelum mengirim.
- **Retry aman**: `429`, `5xx`, `52x` (Cloudflare), dan timeout di-retry dengan
  backoff. `400/401/413/422` tidak di-retry karena payload/token harus diperbaiki.
- **`202` dengan `rejected > 0`** dianggap gagal; perbaiki data, bukan menambah batch.
- **Live stream** membutuhkan ffmpeg/ffprobe dan URL HLS yang bisa diakses dari
  Kaggle. Notebook merekam 30 detik per siklus; jika pemrosesan lebih lama dari 30
  detik, siklus berikutnya otomatis berjalan setelah selesai.
- **Video Drive 1:02:26** diproses per 30 detik. `BASE_TIMESTAMP` wajib diisi dari
  overlay video; jika tidak, timestamp event akan bergeser.
- **Track ID** di-reset per window/siklus. Pipeline penerima memakai `run_id` per
  window dan dedupe berdasarkan `event_id`/`run_id`.
- **Kaggle bukan scheduler.** Untuk production berulang, picu notebook lewat Kaggle
  API/CI, atau pindahkan job ke VM/container dengan cron.
- **Kontrak data.** Events/tracks mengikuti `docs/schema/lane_event.schema.json` dan
  `docs/HANDOFF_DATA_PIPELINE_DASHBOARD.md`.
